In [ ]:
# ── CONFIGURACION DE RUTA ────────────────────────────────────────────
# Ajusta PROJECT_ROOT a la carpeta raiz de tu repositorio
# Ejemplo Windows : r'D:\Documentos\GitHub\dengue'
# Ejemplo Mac/Linux: '/Users/nombre/proyectos/dengue'
# ─────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

PROJECT_ROOT = Path(r'REEMPLAZA_ESTA_RUTA')  # <-- edita aqui

# Agrega src/ al path para poder importar config
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Verifica que la ruta es correcta
assert PROJECT_ROOT.exists(), f'Ruta no encontrada: {PROJECT_ROOT}'
assert (SRC / 'config.py').exists(), 'No se encontro src/config.py — ejecuta primero 00_entorno.ipynb'
print(f'OK: {PROJECT_ROOT}')

# 02b · Regresion Logistica Clinica — OR Ajustados
**Fase:** Analisis multivariado explicativo

Se comparan 4 modelos anidados:
- **M1:** Solo signos de alarma
- **M2:** Signos + sintomas generales
- **M3:** M1 + variables sociodemograficas
- **M4:** M2 + sociodemograficas (modelo completo)

> **Distincion clave:** Estos son modelos **explicativos** (OR con IC95%).
> Los modelos **predictivos** (F1, AUC) estan en el notebook 03.

In [ ]:
import sys
sys.path.insert(0,'../../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
import warnings
from scipy.stats import chi2
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import config
from config import (DATA_INTERIM,OUT_FIGURES,OUT_TABLES,CHECKPOINTS,
    COLORS,TARGET_COL,SIGNOS_ALARMA,SINTOMAS_GENERALES)
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi':150,'savefig.dpi':300,
    'figure.facecolor':'white','axes.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.titlesize':13,'axes.titleweight':'bold'})

df_con = joblib.load(DATA_INTERIM/'datos_con_signos.pkl')
print(f'DCSA cargado: {df_con.shape[0]:,} registros')

In [ ]:
# Preparacion de datos para regresion
# Recodificar clinicas 1=si/0=no (era 1/2 en SIVIGILA)
todas = SINTOMAS_GENERALES + SIGNOS_ALARMA
vars_demo = ['SEXO_','DEF_CLAS_EDAD','TIP_SS_','ESTRATO_']

def prep(df, clinicas, demos=None):
    cols = clinicas + [TARGET_COL] + (demos or [])
    out  = df[cols].copy().dropna()
    for v in clinicas:
        out[v] = out[v].map({1:1, 2:0})
    return out

df_m1 = prep(df_con, SIGNOS_ALARMA)
df_m2 = prep(df_con, todas)
df_m3 = prep(df_con, SIGNOS_ALARMA, vars_demo)
df_m4 = prep(df_con, todas, vars_demo)
print(f'M1:{len(df_m1):,} M2:{len(df_m2):,} M3:{len(df_m3):,} M4:{len(df_m4):,}')

In [ ]:
# Formulas de regresion logistica
def formula(target, clinicas, demos=None):
    terminos = list(clinicas)
    if demos:
        terminos += [f'C({v})' for v in demos]
    return f'{target} ~ ' + ' + '.join(terminos)

specs = [
    ('M1_signos',   formula(TARGET_COL, SIGNOS_ALARMA),          df_m1),
    ('M2_todas',    formula(TARGET_COL, todas),                   df_m2),
    ('M3_sig_demo', formula(TARGET_COL, SIGNOS_ALARMA, vars_demo),df_m3),
    ('M4_completo', formula(TARGET_COL, todas, vars_demo),        df_m4),
]

modelos = {}
for nombre, form, datos in specs:
    m = smf.logit(form, data=datos).fit(disp=False)
    modelos[nombre] = m
    print(f'{nombre}: AIC={m.aic:.1f} | Pseudo-R2={m.prsquared:.3f}')

In [ ]:
# Comparacion de modelos anidados
tabla_comp = pd.DataFrame({
    'Modelo':['M1 Solo signos','M2 Todas clinicas','M3 Signos+demo','M4 Completo'],
    'N_vars':[int(m.df_model) for m in modelos.values()],
    'AIC':   [round(m.aic,2)  for m in modelos.values()],
    'BIC':   [round(m.bic,2)  for m in modelos.values()],
    'PseudoR2':[round(m.prsquared,4) for m in modelos.values()],
})

# LRT M1 vs M2: aportan los sintomas generales informacion adicional?
lr = 2*(modelos['M2_todas'].llf - modelos['M1_signos'].llf)
df_d = modelos['M2_todas'].df_model - modelos['M1_signos'].df_model
p_lr = chi2.sf(lr, df_d)

display(tabla_comp)
tabla_comp.to_csv(OUT_TABLES/'comparacion_modelos_logisticos.csv', index=False)
print(f'LRT M1 vs M2: LR={lr:.2f}, df={df_d:.0f}, p={p_lr:.2e}')
print('=> Sintomas generales aportan informacion significativa' if p_lr<0.05
    else '=> No mejoran significativamente el ajuste')

In [ ]:
# OR ajustados — Modelo M4
m = modelos['M4_completo']
ic = m.conf_int()
tabla_or = pd.DataFrame({
    'Variable':   m.params.index,
    'OR':         np.exp(m.params.values).round(3),
    'IC95_inf':   np.exp(ic[0].values).round(3),
    'IC95_sup':   np.exp(ic[1].values).round(3),
    'p_valor':    m.pvalues.values,
})
tabla_or = tabla_or[tabla_or['Variable'].isin(todas)].copy()
tabla_or['p_fmt'] = tabla_or['p_valor'].apply(
    lambda x: f'{x:.3e}' if x<0.001 else f'{x:.3f}')
tabla_or['Tipo'] = ['Signo de alarma' if v in SIGNOS_ALARMA
    else 'Sintoma general' for v in tabla_or['Variable']]

display(tabla_or.sort_values('OR', ascending=False))
tabla_or.to_csv(OUT_TABLES/'OR_ajustados_M4_DCSA.csv', index=False)

In [ ]:
# Forest plot OR ajustados — M4
# Este grafico muestra la asociacion INDEPENDIENTE de cada variable
df_fp = tabla_or.sort_values('OR').copy()
fig, ax = plt.subplots(figsize=(11,8))
for i, (_, f) in enumerate(df_fp.iterrows()):
    color  = COLORS['con_signos'] if f['Tipo']=='Signo de alarma' else COLORS['sin_signos']
    marker = 'D' if f['p_valor']<0.05 else 'o'
    sz     = 10 if f['p_valor']<0.05 else 7
    ax.errorbar(f['OR'], i,
        xerr=[[f['OR']-f['IC95_inf']],[f['IC95_sup']-f['OR']]],
        fmt=marker, capsize=5, markersize=sz, color=color,
        elinewidth=1.8, markeredgecolor='white', markeredgewidth=0.8)
    ax.text(f['IC95_sup']+0.07, i,
        f"{f['OR']:.2f} [{f['IC95_inf']:.2f}-{f['IC95_sup']:.2f}]",
        va='center', fontsize=8, color='#333333')
ax.set_yticks(range(len(df_fp)))
ax.set_yticklabels(df_fp['Variable'], fontsize=10)
ax.axvline(1, color=COLORS['neutral'], linestyle='--', linewidth=1.5)
ax.set_xlabel('Odds Ratio ajustado (IC 95%)')
ax.set_title('OR ajustados — Modelo completo (M4)\nDengue con signos de alarma')
leyenda = [mpatches.Patch(color=COLORS['con_signos'],label='Signo de alarma'),
    mpatches.Patch(color=COLORS['sin_signos'],label='Sintoma general'),
    plt.Line2D([0],[0],marker='D',color='gray',markersize=8,linewidth=0,label='p<0.05'),
    plt.Line2D([0],[0],marker='o',color='gray',markersize=7,linewidth=0,label='p>=0.05')]
ax.legend(handles=leyenda, fontsize=9, loc='lower right')
ax.set_xlim(0, df_fp['IC95_sup'].max()+1.2)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_FIGURES/'07_forest_OR_ajustados_M4.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# VIF — analisis de colinealidad
# VIF>5 moderado; >10 problematico
X_mat = pd.DataFrame(m.model.exog, columns=m.model.exog_names)
vif_df = pd.DataFrame({
    'Variable': X_mat.columns,
    'VIF': [variance_inflation_factor(X_mat.values, i)
        for i in range(X_mat.shape[1])]
})
vif_df = vif_df[vif_df['Variable']!='Intercept'].sort_values('VIF',ascending=False)
display(vif_df.head(15))

joblib.dump(modelos,  CHECKPOINTS/'02b_modelos_logisticos.pkl')
joblib.dump(tabla_or, CHECKPOINTS/'02b_OR_ajustados_DCSA.pkl')
print('Checkpoints guardados')

## Hallazgos principales

| Variable | OR ajustado M4 | Interpretacion |
|----------|---------------|----------------|
| Caida plaquetas | ~2.7 | Mayor OR independiente |
| Fiebre | ~3.4 | VIF alto, senhal compartida |
| Dolor abdominal | ~1.4 | Asociacion robusta |
| Hemorragia mucosas | ~1.4 | Asociacion robusta |

**Siguiente:** `03_modelado/con_signos/03b_modelos_avanzados_con_signos.ipynb`